# Day 1: PyTorch 핵심 복습 -> HuggingFace로 넘어가기

이 노트북은 오토메타(Autometa) MLOps Engineer 면접을 앞두고, PyTorch를 이미 YOLO/MobileNet 튜토리얼 수준으로 써본 분이 그 아래 깔린 개념들을 한 번 정리하고, 최근 실무에서 많이 쓰는 HuggingFace 생태계로 넘어가는 다리를 놓기 위한 것입니다.

실행 환경: Jetson Orin Nano, JetPack 7.2, PyTorch 설치 직후, GPU 1개.

오늘 다룰 순서:
1. 텐서 기초 (생성 / dtype / device 이동)
2. Autograd (requires_grad, backward)
3. nn.Module + 학습 루프 (forward -> loss -> backward -> optimizer.step)
4. CPU vs GPU 벤치마크 (진짜로 GPU가 빠른지 시간 재보기)
5. HuggingFace Transformers 첫걸음 (작은 Qwen 모델 로드 + 생성)

각 주제는 '개념 설명 -> 따라하기 코드 -> 직접 채우는 TODO -> 연습문제' 순서로 진행됩니다. 코드가 안 되더라도 괜찮으니 일단 끝까지 실행해보고, 안 되는 부분은 solution 노트북과 비교해보세요.

## 1. 텐서 기초: 생성, dtype, device 이동

PyTorch의 모든 데이터는 결국 **텐서(tensor)**입니다. YOLO나 MobileNet을 학습시킬 때 이미 `torch.Tensor`를 수없이 다뤄봤겠지만, 오늘은 한 단계 내려가서 텐서가 가진 세 가지 핵심 속성을 확인합니다.

- **shape**: 몇 차원이고 각 차원의 크기는 얼마인가
- **dtype**: 원소의 자료형 (`float32`가 기본값. edge device에서는 메모리를 아끼려 `float16`도 자주 씁니다)
- **device**: 이 텐서가 실제로 어느 하드웨어 메모리에 올라가 있는가 (`cpu` 또는 `cuda:0`)

참고로 Jetson은 **unified memory** 구조라서 CPU와 GPU가 물리적으로 같은 메모리를 공유합니다 (데스크탑처럼 GPU에 별도 VRAM이 있는 구조가 아닙니다). 그래도 PyTorch 입장에서는 여전히 `cpu` 텐서와 `cuda` 텐서를 구분해서 다룹니다 - 실제 메모리 복사가 최소화되더라도, 어떤 연산 커널(CPU 커널 vs CUDA 커널)을 쓸지가 device로 결정되기 때문입니다.

아래 코드로 GPU가 실제로 잡히는지, 텐서를 옮길 수 있는지 확인해봅시다.

In [ ]:
import torch

print("torch version:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

# 기본은 CPU 텐서, dtype은 float32
x_cpu = torch.tensor([1.0, 2.0, 3.0])
print(x_cpu, x_cpu.dtype, x_cpu.device)

# device 객체를 하나 만들어두고 앞으로 계속 재사용합니다
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_gpu = x_cpu.to(device)
print(x_gpu, x_gpu.dtype, x_gpu.device)

In [ ]:
# TODO: (2, 3) 크기의 랜덤 텐서를 만들고 dtype을 확인한 뒤, device 변수를 이용해 GPU로 옮겨보세요.
# 힌트: torch.randn(행, 열), .dtype, .to(device)

y = ____________  # (2, 3) 랜덤 텐서 생성
print("dtype:", ____________)

y_gpu = ____________  # GPU(또는 device가 cpu라면 그대로)로 이동
print("device:", ____________)

# TODO: y_gpu를 다시 CPU로 되돌리고(.cpu()) numpy 배열로 변환(.numpy())해보세요
z = ____________
print(z)

### 연습문제 1

서로 다른 device에 있는 텐서끼리는 그냥 더할 수 없습니다. 아래 스켈레톤에서:

1. `a`(cpu)와 `b`(cuda, GPU 사용 가능할 때)를 더하는 줄의 주석을 풀어서 실제로 어떤 에러가 나는지 확인하세요.
2. 에러 메시지에서 어떤 문구가 나오는지 읽어보세요 (device mismatch 관련 키워드).
3. device를 맞춰서 정상적으로 덧셈이 되도록 코드를 고쳐보세요.

In [ ]:
a = torch.tensor([1.0, 2.0])              # cpu 텐서
b = torch.tensor([3.0, 4.0]).to(device)   # device 텐서 (GPU가 있다면 cuda)

# 아래 줄의 주석을 풀어서 에러가 나는지 확인해보세요 (device가 cuda일 때만 에러가 납니다)
# result = a + b

# TODO: 에러 없이 더해지도록 아래를 고쳐보세요
result = None
print(result)

## 2. Autograd: 미분을 자동으로 해주는 엔진

딥러닝 학습은 결국 loss를 파라미터로 미분해서 그 반대 방향으로 파라미터를 조금씩 옮기는 과정입니다. PyTorch는 `requires_grad=True`로 표시된 텐서에 대해 수행되는 모든 연산을 기록해뒀다가, `.backward()`가 호출되는 순간 체인룰로 거꾸로 미분값(`.grad`)을 계산합니다. 이것이 **autograd**이고, YOLO 학습시킬 때 `loss.backward()` 한 줄로 이미 써왔던 기능입니다.

가장 간단한 예로 y = x^2 을 미분하면 dy/dx = 2x 입니다. x=3이면 미분값은 6이어야겠죠. 코드로 확인해봅시다.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

print(f"x = {x.item()}, y = {y.item()}")
print(f"dy/dx (x.grad) = {x.grad.item()}")  # 이론값: 2 * 3 = 6

In [ ]:
# TODO: y = x^3 + 2x 를 미분해서 x=2에서의 미분값을 확인하세요.
# 이론값: dy/dx = 3*x^2 + 2 = 3*4 + 2 = 14

x2 = torch.tensor(____________, requires_grad=____________)
y2 = ____________
y2.____________()
print("x2.grad =", ____________)

### 연습문제 2

벡터 입력에 대해서도 autograd가 잘 동작하는지 확인해봅시다. `x3 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)` 에 대해 `z = (x3 ** 2).sum()` 을 계산하고 `z.backward()`를 호출하면, `x3.grad`는 원소별로 `2*x3` (즉 `[2, 4, 6]`)가 나와야 합니다. 직접 확인하고, `backward()`를 한 번 더 호출하면 `x3.grad`가 어떻게 변하는지도 관찰하세요. (grad가 누적되는지 확인하고, 학습 루프에서 왜 `optimizer.zero_grad()`가 필요한지 스스로 설명해보세요.)

In [ ]:
x3 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# TODO: (x3 ** 2).sum() 을 계산해서 z에 저장하세요
z = ____________
z.backward()
print("1번째 backward 이후 grad:", x3.grad)

# TODO: z를 다시 계산하고 backward()를 한 번 더 호출해서 grad가 어떻게 변하는지 확인하세요
____________
____________
print("2번째 backward 이후 grad:", x3.grad)

## 3. nn.Module로 모델 만들고 학습 루프 돌리기

실전에서는 텐서 연산을 하나하나 손으로 쓰지 않고 `nn.Module`(또는 `nn.Linear` 같은 기본 블록)로 모델을 정의합니다. 학습 루프는 어떤 모델이든 사실 다음 4단계의 반복입니다.

1. **forward**: 입력을 모델에 통과시켜 예측값 계산
2. **loss**: 예측값과 정답을 비교해서 손실 계산
3. **backward**: `loss.backward()`로 모든 파라미터의 grad 계산 (2번에서 배운 autograd 그대로)
4. **optimizer.step()**: grad 반대 방향으로 파라미터 업데이트, 그리고 `optimizer.zero_grad()`로 grad 초기화 (안 하면 2번 연습문제에서 본 것처럼 grad가 계속 누적됩니다)

YOLO/MobileNet을 학습시킬 때도 내부적으로 정확히 이 4단계가 반복됐습니다. 여기서는 가장 단순한 문제인 **선형회귀** (`y = 3x + 2`를 흉내내는 직선 찾기)로 이 루프를 처음부터 끝까지 만들어봅니다.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
# 데이터 생성: y = 3x + 2 에 약간의 노이즈를 섞음
x_train = torch.linspace(-5, 5, 100).unsqueeze(1)  # (100, 1)
y_train = 3 * x_train + 2 + torch.randn_like(x_train) * 0.5

model = nn.Linear(in_features=1, out_features=1)  # y = w*x + b, w와 b가 학습 대상
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(200):
    y_pred = model(x_train)          # 1. forward
    loss = loss_fn(y_pred, y_train)  # 2. loss
    optimizer.zero_grad()            # 이전 grad 초기화
    loss.backward()                  # 3. backward
    optimizer.step()                 # 4. 파라미터 업데이트

    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss.item():.4f}")

w, b = model.weight.item(), model.bias.item()
print(f"학습된 w={w:.3f}, b={b:.3f}  (목표: w=3, b=2)")

In [ ]:
# TODO: 위 모델과 데이터를 GPU(device)로 옮겨서 다시 학습해보세요.
# 힌트: 텐서뿐 아니라 model도 .to(device)가 필요합니다.

x_train_gpu = x_train.____________(____________)
y_train_gpu = y_train.____________(____________)

model_gpu = nn.Linear(1, 1).____________(____________)
optimizer_gpu = torch.optim.SGD(model_gpu.parameters(), lr=0.01)

for epoch in range(200):
    y_pred = ____________
    loss = loss_fn(____________, ____________)
    optimizer_gpu.____________()
    loss.____________()
    optimizer_gpu.____________()

print("최종 loss:", loss.item())
print("model_gpu 파라미터 device:", next(model_gpu.parameters()).device)

### 연습문제 3

목표 함수를 `y = -2x + 5` 로 바꿔서 (즉 `y_train`을 새로 만들어서) 모델이 실제로 `w≈-2, b≈5`를 찾아내는지 확인하세요. 그리고 학습률(`lr`)을 0.1로 올리면 어떻게 되는지, 0.0001로 낮추면 어떻게 되는지도 각각 실험하고 loss가 줄어드는 속도가 어떻게 달라지는지 관찰해보세요.

In [ ]:
# TODO: y = -2x + 5 데이터를 새로 만들고, 모델/옵티마이저를 다시 정의해서
# 위의 학습 루프와 같은 구조로 처음부터 학습시켜보세요.

# 1) 데이터 생성


# 2) 모델, loss, optimizer 정의 (lr을 바꿔가며 실험해보세요)


# 3) 학습 루프


## 4. CPU vs GPU 벤치마크: 행렬곱으로 눈으로 확인하기

'GPU가 빠르다'는 말을 그냥 믿기보다 Jetson에서 직접 시간을 재봅시다. 주의할 점이 두 가지 있습니다.

1. **CUDA 연산은 비동기(asynchronous)** 입니다. `torch.matmul(...)`이 리턴됐다고 GPU 연산이 끝난 게 아니라, CPU는 다음 줄로 넘어가고 GPU는 뒤에서 계속 계산 중일 수 있습니다. 정확히 시간을 재려면 `torch.cuda.synchronize()`로 GPU가 실제로 끝날 때까지 기다려야 합니다.
2. **첫 CUDA 호출에는 워밍업 비용**이 있습니다 (CUDA context 초기화, 커널 컴파일/캐싱 등). 그래서 벤치마크 전에 미리 한두 번 연산을 돌려주는 '워밍업'이 필요합니다.

이 두 가지를 지키면서 같은 크기의 행렬곱을 CPU/GPU에서 각각 재서 비교해봅시다.

In [ ]:
import time

size = 2048
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

# CPU 시간 측정
start = time.time()
c_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = time.time() - start
print(f"CPU matmul({size}x{size}): {cpu_time*1000:.2f} ms")

if torch.cuda.is_available():
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    # 워밍업 (첫 CUDA 호출 비용을 벤치마크 구간 밖으로 빼기 위함)
    for _ in range(3):
        _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start = time.time()
    c_gpu = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()  # GPU 연산이 실제로 끝날 때까지 대기
    gpu_time = time.time() - start
    print(f"GPU matmul({size}x{size}): {gpu_time*1000:.2f} ms  (device: {torch.cuda.get_device_name(0)})")
    print(f"speedup: {cpu_time / gpu_time:.1f}x")
else:
    print("GPU를 사용할 수 없어 비교를 생략합니다.")

In [ ]:
# TODO: 아래 함수를 완성해서 주어진 size에 대해 (cpu_time, gpu_time)을 반환하도록 만드세요.
# 워밍업과 torch.cuda.synchronize()를 잊지 마세요!

def benchmark_matmul(size):
    a = torch.randn(size, size)
    b = torch.randn(size, size)

    start = time.time()
    _ = torch.matmul(____________, ____________)
    cpu_time = time.time() - start

    a_gpu = a.to(____________)
    b_gpu = b.to(____________)
    for _ in range(3):
        _ = torch.matmul(____________, ____________)
    torch.cuda.____________()

    start = time.time()
    _ = torch.matmul(____________, ____________)
    torch.cuda.____________()
    gpu_time = time.time() - start

    return cpu_time, gpu_time

cpu_t, gpu_t = benchmark_matmul(1024)
print(f"1024x1024 -> CPU {cpu_t*1000:.2f}ms, GPU {gpu_t*1000:.2f}ms, speedup {cpu_t/gpu_t:.1f}x")

### 연습문제 4

`benchmark_matmul` 함수를 이용해서 크기를 `[128, 256, 512, 1024, 2048]`로 바꿔가며 CPU/GPU 시간과 speedup을 표로 출력해보세요. 크기가 작을 때는 오히려 GPU가 더 느리거나 비슷한 경우도 있는데, 왜 그런지 (커널 실행 오버헤드 vs 실제 연산량 관점에서) 생각해보고 마크다운 셀에 한두 줄로 적어보세요.

In [ ]:
sizes = [128, 256, 512, 1024, 2048]

# TODO: 각 size에 대해 benchmark_matmul을 호출하고 결과를 표 형태로 출력하세요
for s in sizes:
    pass  # 여기를 채우세요

## 5. HuggingFace Transformers 첫걸음

지금까지 한 것 - 텐서, autograd, nn.Module, 학습 루프 - 은 Transformer 모델 안에서도 똑같이 일어나고 있습니다. 다만 매번 어텐션이나 토크나이저를 처음부터 구현하는 건 비효율적이라서, HuggingFace가 이걸 표준화된 라이브러리로 감싸놓았습니다. 생태계를 간단히 정리하면:

- **Transformers**: `AutoModel`, `AutoTokenizer` 같은 클래스로 수많은 모델/토크나이저를 통일된 인터페이스로 불러옵니다.
- **datasets**: 학습/평가용 데이터셋을 다운로드하고 전처리하는 라이브러리.
- **accelerate**: 같은 학습 코드를 단일 GPU든 멀티 GPU든 거의 코드 수정 없이 돌려주는 실행 래퍼.
- **PEFT**: LoRA, QLoRA처럼 '파라미터 일부만 학습하는' 기법들의 구현체.
- **TRL**: `SFTTrainer`, DPO/GRPO 트레이너 등 LLM을 지시-따르기(instruction-following)나 선호도 학습으로 미세조정하는 도구.

오늘은 이 중 가장 기본인 Transformers만 다룹니다. Jetson은 메모리가 넉넉하지 않으니 아주 작은 모델(`Qwen/Qwen2.5-0.5B-Instruct`, 약 0.5B 파라미터)로 시작합니다.

참고로 뒤에서 더 공부하게 될 **LoRA**는 '사전학습된 가중치 W는 그대로 얼려두고, 아주 작은 저랭크 행렬 B·A(r ≪ d)만 학습해서 ΔW를 표현하는' 방법입니다 - fine-tuning으로 인한 가중치 변화가 사실은 저랭크(low-rank)라는 경험적 관찰에 기반합니다. 오늘은 그 전 단계, 즉 사전학습된 모델을 있는 그대로 불러와서 추론(inference)해보는 것까지만 다룹니다.

(만약 `transformers`가 설치되어 있지 않다면 터미널에서 `pip install -U transformers accelerate` 를 먼저 실행하세요.)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # 0.5B 파라미터 정도의 작은 instruct 모델

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Jetson처럼 메모리가 제한적인 환경에서는 fp16 권장
).to(device)
model.eval()

prompt = "Jetson Orin Nano에서 딥러닝을 배우는 재미를 한 문장으로 말해줘."
messages = [{"role": "user", "content": prompt}]

# chat 모델이므로 chat template을 적용해서 모델이 기대하는 입력 포맷으로 변환
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("입력 토큰 개수:", inputs["input_ids"].shape[1])

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
# TODO: 직접 만든 프롬프트로 encode/decode를 호출해보고, 생성 옵션도 바꿔보세요.

my_prompt = "____________"  # 원하는 질문을 한국어나 영어로 적어보세요

# TODO: tokenizer.encode로 my_prompt를 토큰 id 리스트로 변환하세요
token_ids = tokenizer.____________(my_prompt)
print("token ids:", token_ids)

# TODO: 다시 decode해서 원래 문장으로 복원되는지 확인하세요
decoded = tokenizer.____________(____________)
print("decoded:", decoded)

# TODO: max_new_tokens=20, do_sample=True, temperature=0.8 로 바꿔서 생성해보세요
messages2 = [{"role": "user", "content": my_prompt}]
input_text2 = tokenizer.apply_chat_template(messages2, tokenize=False, add_generation_prompt=True)
inputs2 = tokenizer(input_text2, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids2 = model.generate(
        **inputs2,
        max_new_tokens=____________,
        do_sample=____________,
        temperature=____________,
    )
print(tokenizer.decode(output_ids2[0], skip_special_tokens=True))

### 연습문제 5

같은 프롬프트에 대해 텍스트 생성을 CPU에서 한 번, GPU에서 한 번 실행하고 (4번에서 배운 `time` / `torch.cuda.synchronize()`를 그대로 활용) 걸린 시간을 비교해보세요. 모델을 `.to("cpu")`로 옮기면 CPU 추론을 재현할 수 있습니다 (0.5B 모델이라도 CPU에서는 눈에 띄게 느릴 수 있으니 `max_new_tokens`는 20 정도로 작게 주세요).

그리고 오늘 배운 내용을 바탕으로, LoRA를 쓰면 왜 이 모델을 fine-tuning할 때 훨씬 적은 메모리/시간으로 학습할 수 있는지 한두 문장으로 스스로 설명해보세요 (힌트: 3번에서 nn.Linear의 학습 대상 파라미터가 무엇이었는지, LoRA는 그 중 무엇을 학습 대상으로 바꾸는지 떠올려보세요).

In [ ]:
import time

def generate_and_time(model, inputs, max_new_tokens=20):
    # TODO: 4번에서 배운 것처럼, GPU라면 워밍업 + synchronize를 적용해서 생성 시간을 정확히 재보세요
    pass

# TODO: model을 CPU로 옮겨서(.to("cpu")) 같은 prompt로 생성 시간을 재고,
# 다시 GPU로 옮겨서(.to(device)) 생성 시간을 재서 두 값을 비교하세요


## 마무리

오늘 한 것을 순서대로 다시 보면: 텐서(데이터) -> autograd(미분) -> nn.Module + 학습 루프(모델이 데이터로부터 배우는 절차) -> device 이동과 속도 차이(왜 GPU를 쓰는가) -> 이 모든 것 위에 세워진 HuggingFace Transformers(사전학습된 모델을 가져다 쓰기). 이 순서 자체가 '오늘 오토메타 면접에서 PyTorch 기초부터 HuggingFace까지 어떻게 이해하고 있는가'라는 질문에 그대로 답이 됩니다. 아래 면접 Q&A로 오늘 배운 것을 한 번 더 정리해보세요.